# Normalised total energy for a set of runs

One notebook for what used to be two (`EnergyPlotsforDifferentBeta`, `EnergyPlotsforDifferentPpc`, identical apart from
the file list and labels; they are in `archive/quality_checks_superseded/`). Choose the runs with `DATASET`:

* `"beta"`: the two extreme-beta runs (beta_p = 16 and 0.0625)
* `"ppc"`: the four particles-per-cell runs (2048, 1024, 512, 256 ppc; eta = 2.5e-4)

The figure is saved as `EnergyPlot_<DATASET>.png` (both old notebooks saved `EnergyPlotPpc.png`, so they overwrote
each other). The last cell, |B_perp|^2 maps of the four resistivity runs at t = 250, was in both notebooks.

In [ ]:
#All the imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.loading import grid_spacing, load, load_history
from shared.paths import results_dir
from shared.plotting import save_figure, show_field, use_style
from shared.runs import EXTREME_BETA_SCAN, PARTICLES_PER_CELL_SCAN, RESISTIVITY_SCAN, runs_with_labels
use_style()
RESULTS = results_dir("quality_checks", "EnergyPlots")

In [ ]:
DATASET = "ppc"      # "beta" or "ppc"

# runs on the /DATA disk and their legend labels (the scans are defined in shared/runs.py)
DATASETS = {
    "beta": runs_with_labels(EXTREME_BETA_SCAN, group="ExtremeBetaSimulations"),
    "ppc": runs_with_labels(PARTICLES_PER_CELL_SCAN),
}
ENERGY_RANGE = (0.98, 1.02)          # y range of the normalised-energy plot

In [ ]:
# --- Load all runs ---
runs_and_labels = DATASETS[DATASET]
energies = [load_history("EnTot", run) for run, _ in runs_and_labels]
times = [load_history("Time", run) for run, _ in runs_and_labels]

# --- Normalize energies ---
normalized = [energy / energy[0] for energy in energies]

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, max(time_history.max() for time_history in times))
ax.set_ylim(*ENERGY_RANGE)

for time_history, energy_normalised, (run, label) in zip(times, normalized, runs_and_labels):
    ax.plot(time_history, energy_normalised, label=label)

ax.set_xlabel("Time")
ax.set_ylabel("E / E₀")
ax.set_title("Normalized Total Energy vs Time")
ax.grid(True, alpha=0.3)
ax.legend()
save_figure(fig, RESULTS, f"normalised_total_energy_{DATASET}")
plt.show()

In [ ]:
# |B_perp|^2 of each resistivity run
RESISTIVITY_RUNS = [run for run, _ in runs_with_labels(RESISTIVITY_SCAN)]
MAP_TIME = 250

for run in RESISTIVITY_RUNS:
    dx, dy = grid_spacing(run)
    Bx, By = load("Bx", MAP_TIME, run), load("By", MAP_TIME, run)

    fig, ax = plt.subplots(figsize=(10, 8))
    show_field(ax, Bx**2 + By**2, dx, dy)
    ax.set_title(r'$|B_\perp|^2$' + f"  (sim: {run.name}, t = {MAP_TIME})")
    output_path = save_figure(fig, RESULTS, f"B_perp_squared_{run.name}_t{MAP_TIME}")
    plt.close(fig)
    print(f"Saved: {output_path}")